In [4]:
from sklearn.mixture import GaussianMixture
import numpy as np

W_FEAT = np.array([1.0, 0.5, 0.1], dtype=float)   
W_MEAN = np.array([0.6, 0.3, 0.1], dtype=float)   

def fit_gmm(X, random_state=42):
    gmm = GaussianMixture(n_components=2, covariance_type="diag", random_state=random_state)
    Xs = X * np.sqrt(W_FEAT)   
    gmm.fit(Xs)
    return gmm

def pick_depressed_cluster(gmm, X):
    Xs = X * np.sqrt(W_FEAT)
    labels = gmm.predict(Xs)
    scores = []
    for k in range(gmm.n_components):
        idx = labels == k
        if not np.any(idx):
            scores.append(-np.inf)
        else:
            scores.append((X[idx] @ W_MEAN).mean())
    return int(np.argmax(scores))

def predict_dep_prob(gmm, X, dep_cluster):
    Xs = X * np.sqrt(W_FEAT)
    P = gmm.predict_proba(Xs)              
    return P[:, dep_cluster] 

def simple_label(p, high=0.80, low=0.20):
    if p >= high: return "depressed"
    if p <= low:  return "nondepressed"
    return "uncertain"

def predict_one(x, gmm, dep_idx, high=0.80, low=0.20):    
    p = predict_dep_prob(gmm, np.array([x], float), dep_idx)[0]
    label = simple_label(np.array([p]), high=high, low=low)
    return float(p), str(label)

def predict_batch(X_new, gmm, dep_idx, high=0.80, low=0.20):
    probs = predict_dep_prob(gmm, np.array(X_new, float), dep_idx)
    labels = simple_label(probs, high=high, low=low)
    return probs, labels


In [ ]:
import sys
sys.path.append(r"C:\PythonProject\aug-08month_project5")
from AutoPredict import AutoPredict
auto_predict = AutoPredict()

c:\PythonProject\aug-08month_project5\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🚀 Model loaded successfully from C:\PythonProject\aug-08month_project5\jin_sup\model\hwa_in_convnext_base_best_f1_0.7556.pth


c:\PythonProject\aug-08month_project5\.venv\lib\site-packages\torchvision\models\_utils.py:135: UserWarning: Using 'weights' as positional parameter(s) is deprecated since 0.13 and may be removed in the future. Please use keyword parameter(s) instead.
  warnings.warn(


In [ ]:
import pandas as pd
df = pd.read_csv('goon_jip_train.csv')

gmm = fit_gmm(df)
dep_idx = pick_depressed_cluster(gmm, df)     # 어떤 군집이 '우울'인지 결정


In [57]:
def process_pair(png_real_path, wav_real_path):
    png_result = auto_predict.image_predict(png_real_path)["conf"]
    wav_result = auto_predict.sound_predict(wav_real_path)["conf"]
    text_result = auto_predict.text_predict(wav_real_path)["conf"]
    return [png_result, wav_result, text_result]

In [ ]:
import ffmpeg

# input_file = 'C:\\PythonProject\\aug-08month_project5\\jin_sup\\my_test\\sound\\우울증테스트2.m4a'
# output_file = 'C:\\PythonProject\\aug-08month_project5\\jin_sup\\my_test\\sound\\우울증테스트2.wav'

# (
#     ffmpeg
#     .input(input_file)
#     .output(output_file, acodec='pcm_s16le')  # PCM 16-bit 리틀엔디언 포맷 지정
#     .run(overwrite_output=True)
# )

(None, None)

In [60]:
test_sets = [
r"C:\PythonProject\aug-08month_project5\jin_sup\my_test\videos\Jammi\output08.png",
r"C:\PythonProject\aug-08month_project5\jin_sup\my_test\videos\Jammi\output08.wav"
]

result = process_pair(test_sets[0],test_sets[1])
result

tensor([0.1487, 0.8513])
파일에서 변환된 텍스트: 그게 뭐예요


[0.8513374924659729, 0.00031179568084122006, 0.8851132988929749]

In [61]:
p_dep = predict_dep_prob(gmm, [result], dep_idx)  # 우울 확률
labels = [simple_label(p) for p in p_dep]
print(p_dep)   
print(labels)  

[1.56232271e-07]
['nondepressed']


c:\PythonProject\aug-08month_project5\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but GaussianMixture was fitted with feature names
  warnings.warn(


In [62]:
p, lbl = predict_one(result, gmm, dep_idx)  # 단일 샘플
print(p, lbl)  

1.5623227147623998e-07 nondepressed


c:\PythonProject\aug-08month_project5\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but GaussianMixture was fitted with feature names
  warnings.warn(
